In [0]:
%run ../config/config

In [0]:
from pyspark.sql import functions as F
from pathlib import Path
import sys

SRC_PATH = str(Path.cwd().parent / "src")
sys.path.insert(0, SRC_PATH)

from silver import prepare_silver_dataframe, apply_scd_type_2

spark.conf.set("spark.databricks.execution.timeout", "18000")

BRONZE_TABLE_PATH = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_TABLE_PATH = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

In [0]:
bronze_df = spark.table(BRONZE_TABLE_PATH)

latest_ingested_at = (
    bronze_df
    .agg(F.max("ingested_at").alias("latest_ingested_at"))
    .first()["latest_ingested_at"]
)

if latest_ingested_at is None:
    raise ValueError(f"Bronze table {BRONZE_TABLE_PATH} is empty or has no non-null ingested_at values.")

In [0]:
latest_snapshot_df = bronze_df.filter(F.col("ingested_at") == F.lit(latest_ingested_at))

silver_source_df = prepare_silver_dataframe(
    bronze_snapshot_df=latest_snapshot_df,
    columns_to_keep=COLUMNS_TO_KEEP,
    rename_map=RENAME_MAP,
    pnn_categories=PNN_CATEGORIES,
    microelement_columns=MICROELEMENT_COLS,
    country_map=COUNTRY_MAP,
)

apply_scd_type_2(
    spark=spark,
    source_df=silver_source_df,
    target_table_path=SILVER_TABLE_PATH,
    business_key=BUSINESS_KEY,
    columns_to_keep=COLUMNS_TO_KEEP,
    rename_map=RENAME_MAP,
)